## Data Preparation stage:

To conduct the project we need to prepare the raw datasets we collected. The steps are as follows:
- Data filtering: only keeping relevant metrics for our context
- Data cleaning: making sure we got the right data types, handling eventual missing values and duplicates
- Feature engineiring: merging tables and crossing information if necessary

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import os 

### Load the data

In [2]:

# List containing the names of all the files in the data_raw folder :
file_name=[f for f in os.listdir('data_raw') if f.endswith('.csv')]

# Store the dataframes in a dictionnary :

dfs={}

for file in file_name:
    name=file.split('.')[0]

    dfs[name]=pd.read_csv(f'data_raw/{file}')
    print(f"Dataframe {name} has been succesfully imported with {dfs[name].shape[0]} rows and {dfs[name].shape[1]} columns.")


display(dfs.keys())

Dataframe advanced_player_stats has been succesfully imported with 582 rows and 79 columns.
Dataframe advanced_team_stats has been succesfully imported with 30 rows and 46 columns.
Dataframe players_salaries has been succesfully imported with 475 rows and 4 columns.
Dataframe players_stats has been succesfully imported with 582 rows and 67 columns.
Dataframe teams_info has been succesfully imported with 30 rows and 3 columns.
Dataframe team_pergamestats has been succesfully imported with 30 rows and 54 columns.


dict_keys(['advanced_player_stats', 'advanced_team_stats', 'players_salaries', 'players_stats', 'teams_info', 'team_pergamestats'])

### Filtering with only essential rows

#### Team essentials

In [3]:
# Starting team data:
# We select only the relevant columns needed for our analysis and rename them:

team_dim=dfs['teams_info']

team_pergamestats=dfs['team_pergamestats'].loc[:,['TEAM_ID','TEAM_NAME','W','L','W_PCT','FG3_PCT','W_PCT_RANK','FG3_PCT_RANK']]

team_advancedstats=dfs['advanced_team_stats'].loc[:,['TEAM_ID','TEAM_NAME','OFF_RATING','DEF_RATING','NET_RATING','TS_PCT','REB_PCT','OREB_PCT','DREB_PCT','PACE','TM_TOV_PCT','AST_TO','OFF_RATING_RANK','DEF_RATING_RANK','NET_RATING_RANK','TS_PCT_RANK','REB_PCT_RANK','OREB_PCT_RANK','DREB_PCT_RANK','PACE_RANK','TM_TOV_PCT_RANK','AST_TO_RANK']]

#### Players essential

In [4]:
players_pergamestats=dfs['players_stats'].loc[:,['PLAYER_ID','PLAYER_NAME','NICKNAME','TEAM_ID','AGE','GP','W','L','W_PCT','MIN','FG3_PCT','OREB','DREB','REB','AST','TOV','STL','BLK','PTS','TEAM_COUNT']]

players_advancedstats=dfs['advanced_player_stats'].loc[:,['PLAYER_ID','PLAYER_NAME','NICKNAME','TEAM_ID','OFF_RATING','DEF_RATING','NET_RATING','TS_PCT','REB_PCT','OREB_PCT','DREB_PCT','TM_TOV_PCT','AST_TO','USG_PCT','PIE','OFF_RATING_RANK','DEF_RATING_RANK','NET_RATING_RANK','TS_PCT_RANK','REB_PCT_RANK','OREB_PCT_RANK','DREB_PCT_RANK','PACE_RANK','TM_TOV_PCT_RANK','AST_TO_RANK','USG_PCT_RANK','PIE_RANK']]

#### Salaries data

In [5]:
players_salaries=dfs['players_salaries']


### Data inspection and cleanning

In [6]:
# Function to report the full data inspection of a dataframe:
def inspect_df(df):
    print(f"Dataframe has {df.shape[0]} rows and {df.shape[1]} columns.")
    print(f"Columns:{df.columns.tolist()} \nWith data types:{df.dtypes.tolist()}")
    print(f"Missing values per column:\n{df.isnull().sum()}")
    print(f"Duplicated rows:{df.duplicated().sum()}")
    

#### Team Data

In [7]:
# TEAM DATA:
inspect_df(team_dim)


Dataframe has 30 rows and 3 columns.
Columns:['id', 'full_name', 'abbreviation'] 
With data types:[dtype('int64'), <StringDtype(na_value=nan)>, <StringDtype(na_value=nan)>]
Missing values per column:
id              0
full_name       0
abbreviation    0
dtype: int64
Duplicated rows:0


In [8]:
inspect_df(team_pergamestats)

Dataframe has 30 rows and 8 columns.
Columns:['TEAM_ID', 'TEAM_NAME', 'W', 'L', 'W_PCT', 'FG3_PCT', 'W_PCT_RANK', 'FG3_PCT_RANK'] 
With data types:[dtype('int64'), <StringDtype(na_value=nan)>, dtype('int64'), dtype('int64'), dtype('float64'), dtype('float64'), dtype('int64'), dtype('int64')]
Missing values per column:
TEAM_ID         0
TEAM_NAME       0
W               0
L               0
W_PCT           0
FG3_PCT         0
W_PCT_RANK      0
FG3_PCT_RANK    0
dtype: int64
Duplicated rows:0


In [9]:
inspect_df(team_advancedstats)

Dataframe has 30 rows and 22 columns.
Columns:['TEAM_ID', 'TEAM_NAME', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'TS_PCT', 'REB_PCT', 'OREB_PCT', 'DREB_PCT', 'PACE', 'TM_TOV_PCT', 'AST_TO', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'TS_PCT_RANK', 'REB_PCT_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'PACE_RANK', 'TM_TOV_PCT_RANK', 'AST_TO_RANK'] 
With data types:[dtype('int64'), <StringDtype(na_value=nan)>, dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64')]
Missing values per column:
TEAM_ID            0
TEAM_NAME          0
OFF_RATING         0
DEF_RATING         0
NET_RATING         0
TS_PCT             0
REB_PCT            0
OREB_PCT           0
DREB_PCT           0
PACE               0
T

In [7]:
# Data seems clean overall, we adjust the name of team_info columns to match the other dataframes to stay consistent:
team_dim.rename(columns={
    'id':'TEAM_ID',
    'full_name':'TEAM_NAME',
    'abbreviation':'TEAM_ABBREVIATION',
},inplace=True)

team_dim.head(30)

,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION
0,1610612737,Atlanta Hawks,ATL
1,1610612738,Boston Celtics,BOS
2,1610612739,Cleveland Cavaliers,CLE
3,1610612740,New Orleans Pelicans,NOP
4,1610612741,Chicago Bulls,CHI
5,1610612742,Dallas Mavericks,DAL
6,1610612743,Denver Nuggets,DEN
7,1610612744,Golden State Warriors,GSW
8,1610612745,Houston Rockets,HOU
9,1610612746,Los Angeles Clippers,LAC


#### Players Data

In [11]:
inspect_df(players_pergamestats)

Dataframe has 582 rows and 20 columns.
Columns:['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FG3_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'PTS', 'TEAM_COUNT'] 
With data types:[dtype('int64'), <StringDtype(na_value=nan)>, <StringDtype(na_value=nan)>, dtype('int64'), dtype('float64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('int64')]
Missing values per column:
PLAYER_ID      0
PLAYER_NAME    0
NICKNAME       0
TEAM_ID        0
AGE            0
GP             0
W              0
L              0
W_PCT          0
MIN            0
FG3_PCT        0
OREB           0
DREB           0
REB            0
AST            0
TOV            0
STL            0
BLK            0
PTS            0
TEAM_COUNT     0
dtype: int64
Duplicated 

In [12]:
inspect_df(players_advancedstats)

Dataframe has 582 rows and 27 columns.
Columns:['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'TS_PCT', 'REB_PCT', 'OREB_PCT', 'DREB_PCT', 'TM_TOV_PCT', 'AST_TO', 'USG_PCT', 'PIE', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'TS_PCT_RANK', 'REB_PCT_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'PACE_RANK', 'TM_TOV_PCT_RANK', 'AST_TO_RANK', 'USG_PCT_RANK', 'PIE_RANK'] 
With data types:[dtype('int64'), <StringDtype(na_value=nan)>, <StringDtype(na_value=nan)>, dtype('int64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64')]
Missing values per column:
PLAYER_ID          0
PLAYER_NAME        0
NICKNAME

In [8]:
inspect_df(players_salaries)
players_salaries.head()

Dataframe has 475 rows and 4 columns.
Columns:['RK', 'NAME', 'TEAM', 'SALARY'] 
With data types:[dtype('int64'), <StringDtype(na_value=nan)>, <StringDtype(na_value=nan)>, <StringDtype(na_value=nan)>]
Missing values per column:
RK        0
NAME      0
TEAM      0
SALARY    0
dtype: int64
Duplicated rows:0


,RK,NAME,TEAM,SALARY
0,1,"Stephen Curry, G",Golden State Warriors,"$59,606,817"
1,2,"Joel Embiid, C",Philadelphia 76ers,"$55,224,526"
2,3,"Nikola Jokic, C",Denver Nuggets,"$55,224,526"
3,4,"Kevin Durant, F",Houston Rockets,"$54,708,609"
4,5,"Anthony Davis, F",Dallas Mavericks,"$54,126,450"


In [9]:
# We need to adjust the salaries dataframe as the salaries are in string format and the name case contains the player position as well:


players_salaries[['PLAYER_NAME','PLAYER_POSITION']]=players_salaries['NAME'].str.split(',',expand=True)
players_salaries.drop(columns=['NAME'],inplace=True)

players_salaries['SALARY']=players_salaries['SALARY'].str.replace('$','').str.replace(',','').astype(float)
players_salaries.head()

inspect_df(players_salaries)

Dataframe has 475 rows and 5 columns.
Columns:['RK', 'TEAM', 'SALARY', 'PLAYER_NAME', 'PLAYER_POSITION'] 
With data types:[dtype('int64'), <StringDtype(na_value=nan)>, dtype('float64'), <StringDtype(na_value=nan)>, <StringDtype(na_value=nan)>]
Missing values per column:
RK                 0
TEAM               0
SALARY             0
PLAYER_NAME        0
PLAYER_POSITION    0
dtype: int64
Duplicated rows:0


### Working data construction

In [15]:
# Now we can construct the final dataframes by merging the relevant dataframes together and add informations 

In [10]:
# TEAM DATA:
Team_performance=pd.merge(team_pergamestats,team_advancedstats,on=['TEAM_ID','TEAM_NAME'],how='inner',)
Team_performance.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TEAM_ID          30 non-null     int64  
 1   TEAM_NAME        30 non-null     str    
 2   W                30 non-null     int64  
 3   L                30 non-null     int64  
 4   W_PCT            30 non-null     float64
 5   FG3_PCT          30 non-null     float64
 6   W_PCT_RANK       30 non-null     int64  
 7   FG3_PCT_RANK     30 non-null     int64  
 8   OFF_RATING       30 non-null     float64
 9   DEF_RATING       30 non-null     float64
 10  NET_RATING       30 non-null     float64
 11  TS_PCT           30 non-null     float64
 12  REB_PCT          30 non-null     float64
 13  OREB_PCT         30 non-null     float64
 14  DREB_PCT         30 non-null     float64
 15  PACE             30 non-null     float64
 16  TM_TOV_PCT       30 non-null     float64
 17  AST_TO           30 non-null 

In [11]:
# PLAYER DATA:


#-----------------------------Function to normalize the names once the dataset is constructed:
from unidecode import unidecode

def normalize_name(name):
    return unidecode(name)

# Merging the data
Players_performance=pd.merge(players_pergamestats,players_advancedstats,on=['PLAYER_ID','PLAYER_NAME','NICKNAME','TEAM_ID'],how='inner')

# Normalizing names
Players_performance['PLAYER_NAME']=Players_performance['PLAYER_NAME'].apply(normalize_name)

# Inspection
Players_performance.info()

<class 'pandas.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 43 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   PLAYER_ID        582 non-null    int64  
 1   PLAYER_NAME      582 non-null    str    
 2   NICKNAME         582 non-null    str    
 3   TEAM_ID          582 non-null    int64  
 4   AGE              582 non-null    float64
 5   GP               582 non-null    int64  
 6   W                582 non-null    int64  
 7   L                582 non-null    int64  
 8   W_PCT            582 non-null    float64
 9   MIN              582 non-null    float64
 10  FG3_PCT          582 non-null    float64
 11  OREB             582 non-null    float64
 12  DREB             582 non-null    float64
 13  REB              582 non-null    float64
 14  AST              582 non-null    float64
 15  TOV              582 non-null    float64
 16  STL              582 non-null    float64
 17  BLK              582 non-nu

In [23]:
# SALARY DATA:
#Adding player id on salary

Salaries=pd.merge(Players_performance[['PLAYER_ID','PLAYER_NAME','TEAM_ID','GP']],players_salaries[['PLAYER_NAME', 'SALARY']],on='PLAYER_NAME',how='left')

Salaries.info()


<class 'pandas.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PLAYER_ID    582 non-null    int64  
 1   PLAYER_NAME  582 non-null    str    
 2   TEAM_ID      582 non-null    int64  
 3   GP           582 non-null    int64  
 4   SALARY       455 non-null    float64
dtypes: float64(1), int64(3), str(1)
memory usage: 30.4 KB


In [ ]:
# We inspect the patterns of Players with missing salaries

print(f'Number of players with missing salaries: {len(Salaries.loc[Salaries.SALARY.isnull()])}')

print(f'Number of players with missing salaries and less than 20 games played: {len(Salaries.loc[(Salaries.SALARY.isnull()) & (Salaries.GP<20)])}')
display(Salaries.loc[(Salaries.SALARY.isnull()) & (Salaries.GP<20) ] )

print("Players with 50 games played but with missing salary")
display(Salaries.loc[(Salaries.SALARY.isnull()) & (Salaries.GP>=50) ] )

# As most of the players with missing salaries played less than 20 games we can assume they are not consistently part of the rotation 
# Most of them are two-way or on a 10 day contrat 
# We then focus on finding the salary data for important enough players (more than 50 games) and let the other ones as is 





Number of players with missing salaries: 127
Number of players with missing salaries and less than 20 games played: 91


,PLAYER_ID,PLAYER_NAME,TEAM_ID,GP,SALARY
8,1642380,Adama Bal,1610612763,8,NaN
13,1630828,Alex Antetokounmpo,1610612749,6,NaN
15,1631457,Alex Morales,1610612753,4,NaN
18,1631214,Alondes Williams,1610612764,4,NaN
23,1643158,Andersson Garcia,1610612762,5,NaN
...,...,...,...,...,...
549,1631174,Tyler Burton,1610612763,12,NaN
561,1642884,Vladislav Goldin,1610612748,9,NaN
566,1631111,Wendell Moore Jr.,1610612765,6,NaN
572,1642530,Yuki Kawamura,1610612741,18,NaN


Players with 50 games played but with missing salary


,PLAYER_ID,PLAYER_NAME,TEAM_ID,GP,SALARY
4,1628988,Aaron Holiday,1610612745,57,NaN
49,1626171,Bobby Portis Jr.,1610612749,67,NaN
346,1642920,Kobe Sanders,1610612746,68,NaN
474,1641712,Rayan Rupert,1610612763,64,NaN


In [25]:
#--- Decision rule, we retrive the informations for real rotation player as they directly impact the team performance and budget (+50 games so standard contract):


# Retrive the info via hoopshype:

Salaries.loc[4,['SALARY']]=[3080921]
Salaries.loc[49,['SALARY']]=[1344575]
Salaries.loc[346,['SALARY']]=[475497]
Salaries.loc[474,['SALARY']]=[260656]

display(Salaries.loc[Salaries.GP>=50])


Salaries.info()

,PLAYER_ID,PLAYER_NAME,TEAM_ID,GP,SALARY
1,1631260,AJ Green,1610612749,78,2301587.0
4,1628988,Aaron Holiday,1610612745,57,3080921.0
6,1630598,Aaron Wiggins,1610612760,65,10102802.0
7,1642846,Ace Bailey,1610612762,72,9069840.0
9,1641737,Adem Bona,1610612755,71,1955377.0
...,...,...,...,...,...
573,1642274,Yves Missi,1610612740,66,3353040.0
574,1642258,Zaccharie Risacher,1610612737,67,13197720.0
578,1630192,Zeke Nnaji,1610612743,52,8177778.0
579,1630533,Ziaire Williams,1610612751,56,6250000.0


<class 'pandas.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PLAYER_ID    582 non-null    int64  
 1   PLAYER_NAME  582 non-null    str    
 2   TEAM_ID      582 non-null    int64  
 3   GP           582 non-null    int64  
 4   SALARY       459 non-null    float64
dtypes: float64(1), int64(3), str(1)
memory usage: 30.4 KB


## Save the data

In [27]:
Players_performance.to_csv('data_cleaned/Players_performance.csv',index=False)
Team_performance.to_csv('data_cleaned/Team_performance.csv',index=False)
team_dim.to_csv('data_cleaned/Team_dim.csv',index=False)
Salaries.to_csv('data_cleaned/Players_salaries.csv',index=False)